# ERP SQL Agent — Walkthrough

Ask a manufacturing ERP database questions in plain English. Three ways to turn a question into SQL
are compared:

| Version | Based on | Idea |
|---|---|---|
| **A** chain | LangChain `create_sql_query_chain` (*AI Agents and Applications*, ch10) | question → SQL → run. One try. |
| **B** toolkit agent | LangChain `SQLDatabaseToolkit` (ch11) | an agent that looks at tables, checks and runs queries |
| **C** grounded agent | this project | reads the **real schema + what each column means**, and every query is **validated with code** before it runs |

Every output below is real. **Sections:** 1. The database · 2. Schema tools · 3. The validator ·
4. One question, three versions · 5. A question the data can't answer · 6. The MCP server ·
7. Evaluation · 8. Findings

Live Claude calls in this notebook: about 20 (Claude Haiku 5.5).

In [1]:
import json, logging, os, sqlite3, subprocess, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
if not (ROOT / "data/erp.db").exists():
    subprocess.run([sys.executable, "data/build_db.py"], check=True)

from IPython.display import Markdown, display
from erp_agent.config import DB_PATH

MODEL = "claude-haiku-5-5"      # the small model: where the differences between A, B and C show most

def table(rows, headers):
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))

## 1. The database

A **synthetic** ERP for a made-up implement manufacturer, *Prairie Implements*, built by
`data/build_db.py` with a fixed random seed (same data every time). Nothing comes from a real
company. 9 tables: what we buy, make, stock and sell.

In [2]:
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
d = json.load(open(ROOT / "data/data_dictionary.json"))
table([[t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], d[t]["description"]] for t in tables],
      ["Table", "Rows", "What it holds"])

| Table | Rows | What it holds |
|---|---|---|
| customers | 30 | Dealers and farms that buy from us. |
| inventory | 36 | Stock per part per warehouse. |
| parts | 20 | Every item we buy, make or sell. Finished machines are parts with category 'Finished good'. |
| po_lines | 105 | Lines of a purchase order: which part, how many. |
| purchase_orders | 60 | Orders we place with suppliers (header). |
| sales_orders | 120 | Orders customers place with us (header). |
| so_lines | 178 | Lines of a sales order: which machine, how many, at what price. |
| suppliers | 6 | Companies we buy parts and raw material from. |
| work_orders | 40 | Production jobs: build a quantity of a finished good. |

The **data dictionary** says what each column *means*: the part a plain schema can't. For example
`inventory`: a column type says `INTEGER`; the dictionary says *available = on hand − allocated*.

In [3]:
table([[c, desc] for c, desc in d["inventory"]["columns"].items()], ["inventory column", "Meaning"])
print("sample rows:")
for r in con.execute("SELECT * FROM inventory LIMIT 4"): print("  ", r)

| inventory column | Meaning |
|---|---|
| part_id | The part. |
| warehouse | Warehouse: 'Main plant' or 'Parts depot'. |
| qty_on_hand | Units physically in the warehouse. |
| qty_allocated | Units already promised to orders. Available = qty_on_hand - qty_allocated. |

sample rows:
   (1, 'Main plant', 286, 20)
   (2, 'Main plant', 257, 85)
   (3, 'Main plant', 119, 4)
   (4, 'Main plant', 3901, 1093)


A question the evaluation asks, answered directly in SQL: **which parts are below their reorder point,
counting available stock across warehouses?** (6 parts; remember the number.)

In [4]:
rows = con.execute('''SELECT p.part_number, p.description, p.reorder_point,
       SUM(i.qty_on_hand - i.qty_allocated) AS available
FROM parts p JOIN inventory i ON i.part_id = p.part_id
GROUP BY p.part_id HAVING available < p.reorder_point''').fetchall()
table(rows, ["Part", "Description", "Reorder point", "Available"])

| Part | Description | Reorder point | Available |
|---|---|---|---|
| PI-1002 | Steel tube 3in | 300 | 172 |
| PI-1003 | Angle iron 2in | 250 | 115 |
| PI-1007 | Gearbox 540rpm | 20 | 8 |
| PI-1015 | Paint, implement red (L) | 120 | 95 |
| PI-1019 | Post pounder PP-200 | 3 | 1 |
| PI-1020 | Rock picker RP-12 | 3 | 2 |

## 2. The grounded agent's schema tools

Version C gets three tools. The first two are how it **grounds** itself before writing SQL:

- `list_tables()`: every table with a one-line description
- `get_schema(tables)`: each column's type **and meaning**, plus 3 sample rows
- `run_sql(sql)`: the only way to touch data, validated first (next section)

In [5]:
from erp_agent.schema import list_tables_text, schema_text
print(list_tables_text())
print()
print(schema_text(["parts"]))

- customers: Dealers and farms that buy from us.
- inventory: Stock per part per warehouse.
- parts: Every item we buy, make or sell. Finished machines are parts with category 'Finished good'.
- po_lines: Lines of a purchase order: which part, how many.
- purchase_orders: Orders we place with suppliers (header).
- sales_orders: Orders customers place with us (header).
- so_lines: Lines of a sales order: which machine, how many, at what price.
- suppliers: Companies we buy parts and raw material from.
- work_orders: Production jobs: build a quantity of a finished good.

TABLE parts -- Every item we buy, make or sell. Finished machines are parts with category 'Finished good'.
  part_id INTEGER  -- Internal part number (use for joins).
  part_number TEXT  -- Part number printed on labels, e.g. 'PI-1001'.
  description TEXT  -- What the part is.
  category TEXT  -- 'Raw material', 'Component', 'Hydraulics', 'Electrical' or 'Finished good'.
  unit_cost REAL  -- Standard cost per unit in CAD

## 3. The validator: checked by code before anything runs

`erp_agent/validator.py` uses `sqlglot` to parse the SQL and check it against the **real** schema:

1. it parses as SQLite
2. exactly one statement, and a read-only `SELECT`
3. every table exists
4. every column exists in the table it's used with (`sqlglot`'s `qualify` traces each alias)
5. a `LIMIT` is added if missing

Each rejection message says what's wrong **and what exists**, so an agent can fix its query.

In [6]:
from erp_agent.validator import validate_sql
tests = [
    "SELECT name, country FROM suppliers",
    "SELECT phone FROM suppliers",
    "SELECT * FROM employees",
    "DELETE FROM parts",
    "SELECT 1; DROP TABLE parts",
    "WITH s AS (SELECT part_id, SUM(qty_on_hand) q FROM inventory GROUP BY part_id) "
    "SELECT p.part_number, s.q FROM parts p JOIN s ON s.part_id = p.part_id",
]
rows = []
for sql in tests:
    v = validate_sql(sql)
    rows.append([f"`{sql[:60]}`", "✓ runs" if v.ok else "✗ rejected", (v.errors[0] if v.errors else v.sql)[:110]])
table(rows, ["Query", "Result", "Message / SQL that runs"])

| Query | Result | Message / SQL that runs |
|---|---|---|
| `SELECT name, country FROM suppliers` | ✓ runs | SELECT name, country FROM suppliers LIMIT 200 |
| `SELECT phone FROM suppliers` | ✗ rejected | Column 'phone' could not be resolved. Line: 1, Col: 12. Columns that exist -> suppliers: supplier_id, name, co |
| `SELECT * FROM employees` | ✗ rejected | Unknown table(s): employees. Existing tables: customers, inventory, parts, po_lines, purchase_orders, sales_or |
| `DELETE FROM parts` | ✗ rejected | Only read-only SELECT queries are allowed. |
| `SELECT 1; DROP TABLE parts` | ✗ rejected | Send exactly one SQL statement. |
| `WITH s AS (SELECT part_id, SUM(qty_on_hand) q FROM inventory` | ✓ runs | WITH s AS (SELECT part_id, SUM(qty_on_hand) AS q FROM inventory GROUP BY part_id) SELECT p.part_number, s.q FR |

The query is also run on a **read-only** connection (`mode=ro`): a second safety layer, so even a
query that slipped past the validator couldn't change data.

## 4. One question, three versions

*"Which parts are below their reorder point, and who supplies them?"* (live calls, Claude Haiku).

In [7]:
from erp_agent.versions import run_chain, run_toolkit_agent, run_grounded_agent
Q = "Which parts are below their reorder point, and who supplies them?"

results = {}
for name, fn in [("A chain", run_chain), ("B toolkit", run_toolkit_agent), ("C grounded", run_grounded_agent)]:
    r = fn(Q, MODEL)
    results[name] = r
    print(f"=== {name}: {r.seconds:.1f}s, {r.queries_tried} queries run, {r.rejected} rejected")
    if r.error:
        print("   ERROR:", r.error.splitlines()[0][:200])
    for i, sql in enumerate(r.sqls, 1):
        n = len(con.execute(sql).fetchall())
        print(f"   query {i} -> {n} rows:  {' '.join(sql.split())[:260]}")
    print()

=== A chain: 7.3s, 1 queries run, 0 rejected
   query 1 -> 3 rows:  SELECT p."part_id", p."part_number", p."description", p."reorder_point", SUM(i."qty_on_hand") AS "total_on_hand", s."name" AS "supplier_name" FROM parts p JOIN inventory i ON i."part_id" = p."part_id" JOIN suppliers s ON s."supplier_id" = p."supplier_id" GROUP



=== B toolkit: 7.1s, 2 queries run, 0 rejected
   query 1 -> 6 rows:  SELECT p.part_id, p.part_number, p.description, p.reorder_point, SUM(i.qty_on_hand) AS on_hand, SUM(i.qty_allocated) AS allocated, SUM(i.qty_on_hand - i.qty_allocated) AS available, s.name AS supplier, s.lead_time_days FROM parts p JOIN inventory i ON i.part_i
   query 2 -> 0 rows:  SELECT p.part_id, p.part_number, p.description, p.reorder_point, s.name FROM parts p LEFT JOIN suppliers s ON s.supplier_id = p.supplier_id WHERE p.part_id NOT IN (SELECT part_id FROM inventory)



=== C grounded: 9.0s, 2 queries run, 0 rejected
   query 1 -> 4 rows:  SELECT p.part_id, p.part_number, p.description, p.category, p.reorder_point, SUM(i.qty_on_hand - i.qty_allocated) AS total_available, s.name AS supplier, s.country, s.lead_time_days FROM parts p JOIN inventory i ON i.part_id = p.part_id LEFT JOIN suppliers s O
   query 2 -> 2 rows:  SELECT p.part_id, p.part_number, p.description, p.reorder_point, SUM(i.qty_on_hand - i.qty_allocated) AS total_available FROM parts p JOIN inventory i ON i.part_id = p.part_id WHERE p.category = 'Finished good' GROUP BY p.part_id HAVING total_available < p.reo



Compare each version with the **6 parts** in section 1. The chain (A) gets one try: LangChain's
`create_sql_query_chain` also adds **`LIMIT 5`** by default, so on list questions it can never show more
than 5 rows, and a wrong formula or join can't be corrected. The agents (B, C) can run follow-up queries
and look at the results before answering.

Version C's **trace**: list the tables, read the schema (with column meanings), run validated queries,
then answer from the rows:

In [8]:
for step in results["C grounded"].trace:
    print(step[:170])
print("\nAnswer:\n" + results["C grounded"].answer[:1200])

call list_tables({})
  -> '- customers: Dealers and farms that buy from us.\n- inventory: Stock per part per warehouse.\n- parts: Every item we buy, make or sell. Finished machines are part'
call get_schema({'tables': ['inventory', 'parts', 'suppliers', 'purchase_orders', 'po_lines']})
  -> "TABLE inventory -- Stock per part per warehouse.\n  part_id INTEGER  -- The part.\n  warehouse TEXT  -- Warehouse: 'Main plant' or 'Parts depot'.\n  qty_on_hand IN"
call run_sql({'sql': "SELECT p.part_id, p.part_number, p.description, p.category, p.reorder_point,\n  SUM(i.qty_on_hand - i.qty_allocated) AS total_available,\n  s.name A
  -> '4 row(s)\npart_id | part_number | description | category | reorder_point | total_available | supplier | country | lead_time_days\n2 | PI-1002 | Steel tube 3in | R'
call run_sql({'sql': "SELECT p.part_id, p.part_number, p.description, p.reorder_point,\n  SUM(i.qty_on_hand - i.qty_allocated) AS total_available\nFROM parts p\nJOIN inve
  -> '2 row(s)\npart_id | part

## 5. A question the data can't answer

*"Which sales rep handled the most orders?"*: there is no sales-rep column anywhere. A good system
says so; a bad one answers a **different** question that looks plausible.

In [9]:
Q2 = "Which sales rep handled the most orders?"
a = run_chain(Q2, MODEL)
print("A chain SQL:   ", a.sql)
print("A chain result:", (a.answer or a.error)[:200])
c = run_grounded_agent(Q2, MODEL)
print("\nC grounded:    ", c.answer[:400])

A chain SQL:    SELECT "customer_id", COUNT(*) AS "order_count" FROM "sales_orders" GROUP BY "customer_id" ORDER BY "order_count" DESC LIMIT 5;
A chain result: [(20, 8), (10, 8), (5, 8), (22, 6), (15, 6)]



C grounded:     The database doesn't record which sales rep handled each order. The sales tables (`sales_orders`, `so_lines`) and the `customers` table have no sales rep field. `sales_orders` has the customer, dates, and status, but nothing identifying a rep, so I can't say who handled the most orders.

If rep assignments are kept somewhere else, such as a CRM or a spreadsheet, that data would need to be loaded i


## 6. The MCP server

`mcp_server.py` exposes the same three tools to any MCP client (Claude Code, the Inspector…). The
client's own model writes the SQL; the server grounds and validates it. A rejected query raises
`ToolError`, so the model sees the message. Below, a raw stdio session:

In [10]:
msgs = [
    {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18",
     "capabilities": {}, "clientInfo": {"name": "notebook", "version": "1"}}},
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "run_sql",
     "arguments": {"sql": "SELECT phone FROM suppliers"}}},
    {"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "run_sql",
     "arguments": {"sql": "SELECT country, COUNT(*) FROM suppliers GROUP BY country"}}},
]
script = "import time\n" + "".join(f"print({json.dumps(json.dumps(m))}, flush=True)\n" for m in msgs) + "time.sleep(15)\n"
feeder = subprocess.Popen([sys.executable, "-c", script], stdout=subprocess.PIPE)
server = subprocess.Popen([sys.executable, str(ROOT / "mcp_server.py")], stdin=feeder.stdout,
                          stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, cwd=ROOT)
replies = {}
while len(replies) < 4:
    line = server.stdout.readline()
    if not line:
        break
    m = json.loads(line)
    if "id" in m:
        replies[m["id"]] = m.get("result", m.get("error"))
server.kill(); feeder.kill()
print("tools:", [t["name"] for t in replies[2]["tools"]])
print("\nrun_sql('SELECT phone FROM suppliers') -> isError =", replies[3].get("isError"))
print("  ", replies[3]["content"][0]["text"][:220])
print("\nrun_sql(valid query) ->")
print(replies[4]["content"][0]["text"])

tools: ['list_tables', 'get_schema', 'run_sql']

run_sql('SELECT phone FROM suppliers') -> isError = True
   Error executing tool run_sql: REJECTED: Column 'phone' could not be resolved. Line: 1, Col: 12. Columns that exist -> suppliers: supplier_id, name, country, lead_time_days

run_sql(valid query) ->
3 row(s)
country | COUNT(*)
Canada | 3
Germany | 1
USA | 2


## 7. Evaluation

### The test set

25 questions in four levels, each with the correct answer written as **SQL** (the "gold" query).
Three are **unanswerable** (phone number, sales rep, labour hours): the right answer is to say the data
isn't stored.

In [11]:
questions = [json.loads(l) for l in open(ROOT / "eval/questions.jsonl")]
from collections import Counter
print(dict(Counter(q["level"] for q in questions)))
table([[q["id"], q["level"], q["question"][:85], f"`{(q['gold_sql'] or '— (unanswerable)')[:60]}`"]
       for q in questions[::4]], ["id", "Level", "Question", "Gold SQL"])

{'easy': 6, 'medium': 8, 'hard': 8, 'unanswerable': 3}


| id | Level | Question | Gold SQL |
|---|---|---|---|
| q01 | easy | How many customers do we have? | `SELECT COUNT(*) FROM customers` |
| q05 | easy | Which supplier has the longest lead time? | `SELECT name FROM suppliers ORDER BY lead_time_days DESC LIMI` |
| q09 | medium | How many customers do we have in each province? | `SELECT province, COUNT(*) FROM customers GROUP BY province` |
| q13 | medium | Which work orders are due in October 2026 and not complete? | `SELECT wo_id FROM work_orders WHERE due_date BETWEEN '2026-1` |
| q17 | hard | For parts below their reorder point (available stock across warehouses), who is the s | `SELECT p.part_number, s.name FROM parts p JOIN suppliers s O` |
| q21 | hard | On average, how many days did shipped orders take from order date to ship date? | `SELECT ROUND(AVG(julianday(ship_date)-julianday(order_date))` |
| q25 | unanswerable | How many hours of labour did work order 5 take? | `— (unanswerable)` |

### How an answer is scored (`eval/run_eval.py`)

Compare **results, not SQL text**: different SQL can be equally right.

- run the agent's queries and the gold query; correct if any query the agent ran returns the same rows
  (extra columns allowed; ranking questions check the top answer; decimals rounded to 1 place)
- for a single-value answer, also correct if that value appears in the written answer ("**30** open orders")
- unanswerable: correct only if the answer says the data isn't stored; substituting another column fails

In [12]:
sys.path.insert(0, str(ROOT / "eval"))
import run_eval as E
gold = next(q for q in questions if q["id"] == "q08")["gold_sql"]
cases = [
    ("same rows + extra columns", gold.replace("SELECT p.part_number", "SELECT p.part_number, p.description")),
    ("cut off at 5 rows (version A)", gold + " LIMIT 5"),
]
table([[name, "✓" if E.same_result(sql, gold) else "✗"] for name, sql in cases], ["Predicted query (q08)", "Scored"])
print("refusal detected:", bool(E.REFUSAL.search("The database doesn't record labour hours for work orders.")))
print("substitution detected as refusal:", bool(E.REFUSAL.search("Midwest Hydraulics is in the USA, lead time 21 days.")))

| Predicted query (q08) | Scored |
|---|---|
| same rows + extra columns | ✓ |
| cut off at 5 rows (version A) | ✗ |

refusal detected: True
substitution detected as refusal: False


### Results: two models, three versions

In [13]:
rows = []
for model in ["claude-sonnet-5-5", "claude-haiku-5-5"]:
    S = json.load(open(ROOT / f"eval/results/{model}/summary.json"))["summary"]
    for v, name in [("A", "A chain"), ("B", "B toolkit"), ("C", "C grounded")]:
        s, L = S[v], S[v]["by_level"]
        rows.append([model.replace("claude-", ""), name, f"**{s['correct_%']:.0f}%**", L["easy"], L["medium"], L["hard"],
                     L["unanswerable"], f"{s['invented_query_%']:.0f}%", f"{s['avg_seconds']} s",
                     s["avg_tokens"] or "n/a"])
table(rows, ["Model", "Version", "Correct", "Easy", "Medium", "Hard", "Unanswerable", "Invented names", "Avg time", "Avg tokens"])

| Model | Version | Correct | Easy | Medium | Hard | Unanswerable | Invented names | Avg time | Avg tokens |
|---|---|---|---|---|---|---|---|---|---|
| sonnet-5-5 | A chain | **92%** | 6/6 | 7/8 | 8/8 | 2/3 | 0% | 6.2 s | n/a |
| sonnet-5-5 | B toolkit | **100%** | 6/6 | 8/8 | 8/8 | 3/3 | 0% | 6.6 s | 7011 |
| sonnet-5-5 | C grounded | **100%** | 6/6 | 8/8 | 8/8 | 3/3 | 0% | 6.0 s | 5579 |
| haiku-5-5 | A chain | **64%** | 6/6 | 6/8 | 4/8 | 0/3 | 0% | 4.3 s | n/a |
| haiku-5-5 | B toolkit | **100%** | 6/6 | 8/8 | 8/8 | 3/3 | 0% | 4.8 s | 7425 |
| haiku-5-5 | C grounded | **100%** | 6/6 | 8/8 | 8/8 | 3/3 | 0% | 4.2 s | 5876 |

Why the chain (A) failed with Haiku, question by question:

In [14]:
res = json.load(open(ROOT / "eval/results/claude-haiku-5-5/results_A.json"))
for r in res:
    if not r["correct"]:
        why = ("crashed: " + r["error"].splitlines()[0][:70]) if r["error"] else \
              ("cut off at 5 rows" if r["pred_sql"] and "LIMIT 5" in r["pred_sql"].upper() else "answered a different question")
        print(f"{r['id']} ({r['level']:12}) {why:45} | {r['question'][:60]}")

q08 (medium      ) cut off at 5 rows                             | Which parts are below their reorder point, counting availabl
q11 (medium      ) cut off at 5 rows                             | Which purchase orders are partially received? Give their num
q17 (hard        ) cut off at 5 rows                             | For parts below their reorder point (available stock across 
q18 (hard        ) cut off at 5 rows                             | How many units are still to be received on open or partially
q20 (hard        ) crashed: OperationalError: near "```sql       | What is the gross margin per unit (list price minus unit cos
q22 (hard        ) crashed: OperationalError: near "```sql       | On orders that were not cancelled, do dealers or direct farm
q23 (unanswerable) cut off at 5 rows                             | What is the phone number of Midwest Hydraulics?
q24 (unanswerable) answered a different question                 | Which sales rep handled the most orders?
q25 (unan

## 8. Findings

**1. The standard chain is the weak link, and it gets worse on the small model** (Sonnet 92% →
Haiku 64%). It fails in three ways:
- **a silent 5-row cap:** `create_sql_query_chain` adds `LIMIT 5`, so "which parts are low?" shows 5 of 6;
- **crashing on formatting:** the SQL comes back wrapped in ```` ```sql ```` fences despite the cleaning prompt, with no retry;
- **substituting instead of refusing:** "which sales rep…?" returns the top *customer*. It looks
  plausible but answers a different question: the most dangerous kind of error.

**2. Agents that look at the data and can retry got 100% on both models.** With the grounded agent,
the small, cheap model matches the expensive one, and is the fastest setup (~4 s per question).

**3. C is leaner than B:** about 20% fewer tokens and 26–37% fewer queries, because column meanings
arrive up front and it explores less.

**4. Honest finding: no version invented a table or column name**, and C's validator never had to
reject a query. On a clean 9-table schema with a dictionary, modern Claude models don't hallucinate
names. Here the validator is a safety net (read-only, single SELECT, capped rows, readable errors) that
hasn't been needed yet. **Next experiment:** cryptic ERP-style names (`QTY_ORD`, `PRT_NO`) and a much
larger schema, where invented names appear and the validator has to earn its place.

**5. Evaluating the evaluator.** The first scoring run gave A 68%, B 60%, C 68%, but about half the
failures were scoring bugs: only the last query was checked, extra columns broke the comparison, ranked
lists were marked wrong, and refusals like "doesn't record" were missed. Each fix was tested on known
right and wrong cases before re-scoring. (The strict first log is kept in `eval/eval_log_v1_strict_scoring.txt`.)